# Predicting the Electronic Band Gap of Inorganic Crystalline Materials from Composition and Structure
### Stage 7 — Evaluation, Model Comparison and Model Selection

Reads the saved Stage 6 outputs (cross-validation results, final-fit train/test metrics, and the held-out test predictions). **Nothing is re-fitted here and the test set is not used to change any model.** This notebook:

1. Builds the clean comparison table (Stage 2 layout)
2. Quantifies overfitting (train vs. CV vs. test)
3. Selects the model to carry forward, using a stated rule based on cross-validation
4. Puts **uncertainty intervals** on the test results (bootstrap over formulas)
5. Asks, with those intervals, whether structural descriptors add predictive value
6. Summarises cost, robustness and interpretability, and the effect of the split strategy

**Not in this notebook:** feature importance, SHAP, and error breakdowns by chemistry (Stage 8), and the chemical interpretation (Stage 9).


## Setup and load the Stage 6 outputs


In [ ]:
import os
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
N_BOOT = 1000

DRIVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"
STAGE6_FILES = [
    "stage6_model_results.csv",
    "stage6_test_predictions.csv",
    "stage6_random_vs_grouped.csv",
    "stage6_config.json",
]

def ensure_local(name):
    if os.path.exists(name):
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass
    drive_path = os.path.join(DRIVE_DIR, name)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, name)
        print(f"Loaded {name} from Google Drive.")
    else:
        raise FileNotFoundError(
            f"{name} not found locally or at {drive_path}. Make sure Stage 6 finished and its "
            "final cell copied its outputs to that Drive folder."
        )

for fname in STAGE6_FILES:
    ensure_local(fname)

results = pd.read_csv("stage6_model_results.csv")
# keep_default_na=False so the formula 'NaN' (sodium nitride) is not read as a missing value
preds = pd.read_csv("stage6_test_predictions.csv", keep_default_na=False, na_values=[""])
rvg = pd.read_csv("stage6_random_vs_grouped.csv")
with open("stage6_config.json") as f:
    config = json.load(f)

assert len(results) == 10, f"Expected 10 model configurations, found {len(results)}"

print("Stage 6 run configuration:")
for k in ["random_seed", "fast_mode", "cv_folds", "n_trees", "n_train_rows", "n_test_rows",
          "n_rows_with_missing_features", "random_split_formula_overlap_fraction"]:
    print(f"  {k}: {config[k]}")
assert config["fast_mode"] is False, "These results came from a FAST_MODE trial run -- do not report them."


## 1. Model comparison table

Cross-validation columns come from grouped 5-fold CV on the training portion. Test columns come from the single evaluation on the held-out, formula-grouped test set. All models are scored on the same complete-case rows. Rows are sorted by CV MAE within each feature set.


In [ ]:
MODEL_ORDER = ["Dummy (mean)", "Ridge", "HistGradientBoosting", "RandomForest", "ExtraTrees"]
FEATURE_SETS = ["composition_only", "composition_structure"]

def comparison_table(fs):
    d = results[results.feature_set == fs].sort_values("CV_MAE")
    out = d[["model", "CV_MAE", "CV_RMSE", "CV_R2", "Test_MAE", "Test_RMSE", "Test_R2"]].copy()
    out.columns = ["Model", "CV MAE", "CV RMSE", "CV R2", "Test MAE", "Test RMSE", "Test R2"]
    return out.reset_index(drop=True)

pd.set_option("display.width", 200)
for fs in FEATURE_SETS:
    print(f"\n=== Feature set: {fs} (errors in eV) ===")
    print(comparison_table(fs).round(4).to_string(index=False))


## 2. Overfitting: training vs. cross-validation vs. test

A model that scores far better on the data it was trained on than on data it hasn't seen is fitting noise or memorising specifics of the training set (overfitting). The gaps below quantify this. A model can overfit and still be the best held-out performer, so the gap is a diagnostic to discuss, not by itself a reason to reject a model.

- **MAE gap** = Test MAE − Train MAE (larger = worse generalisation)
- **R2 gap** = Train R2 − Test R2
- **CV vs test** = Test MAE − CV MAE. A value near zero means the CV estimate predicted the test result well, which is what an untouched test set should show.


In [ ]:
ov = results[["feature_set", "model", "Train_MAE", "CV_MAE", "Test_MAE",
              "Train_R2", "CV_R2", "Test_R2"]].copy()
ov["MAE_gap_test_minus_train"] = ov["Test_MAE"] - ov["Train_MAE"]
ov["R2_gap_train_minus_test"] = ov["Train_R2"] - ov["Test_R2"]
ov["MAE_test_minus_cv"] = ov["Test_MAE"] - ov["CV_MAE"]
ov = ov.sort_values(["feature_set", "R2_gap_train_minus_test"], ascending=[True, False]).reset_index(drop=True)

print(ov.round(4).to_string(index=False))
ov.to_csv("stage7_overfitting_table.csv", index=False)


## 3. Model selection rule

**Rule: the model with the lowest mean grouped-CV MAE across all 10 configurations.** Selection uses cross-validation only. The test set is reported for the selected model but does not choose it. MAE is used because it is in eV, is easy to interpret, and is less dominated by the heavy-tailed errors than RMSE.

Below we also report how clearly the top configurations are separated relative to fold-to-fold variation, so the choice isn't presented as more decisive than it is.


In [ ]:
ranked = results.sort_values("CV_MAE").reset_index(drop=True)
print(ranked[["feature_set", "model", "CV_MAE", "CV_MAE_std", "CV_R2", "fit_seconds"]].head(5).round(4).to_string())

best, runner = ranked.iloc[0], ranked.iloc[1]
SELECTED = (best["feature_set"], best["model"])
gap = runner["CV_MAE"] - best["CV_MAE"]

print(f"\nSelected by CV rule: {SELECTED[1]} on {SELECTED[0]}")
print(f"Runner-up:           {runner['model']} on {runner['feature_set']}")
print(f"CV MAE gap (runner-up minus selected): {gap:.4f} eV")
print(f"Fold-to-fold std of CV MAE: selected {best['CV_MAE_std']:.4f}, runner-up {runner['CV_MAE_std']:.4f}")
if gap < max(best["CV_MAE_std"], runner["CV_MAE_std"]):
    print("-> The runner-up is within one fold-to-fold standard deviation: the two are not clearly separated by CV alone.")
else:
    print("-> The gap exceeds one fold-to-fold standard deviation.")


## 4. Uncertainty on the test results (bootstrap over formulas)

The test set is one finite sample, so each test metric has sampling uncertainty. We estimate it with a **cluster bootstrap**: formulas (not individual rows) are resampled with replacement, because polymorphs of one formula are not independent observations. The same resamples are applied to every model, so differences between models are **paired**.

Before bootstrapping, the notebook recomputes every test metric from the saved predictions and checks it against the Stage 6 values, as an integrity check.

**What this interval does and does not cover:** it captures variability from *which compounds happen to be in the test set*, for these already-fitted models. It does **not** capture variability from refitting, from a different train/test split, or from the random seeds of the tree models.


In [ ]:
configs = [(r.feature_set, r.model) for r in results.itertuples()]

def get_pred(fs, model):
    return preds[(preds.feature_set == fs) & (preds.model == model)].set_index("material_id")

base = get_pred("composition_structure", "Dummy (mean)")
ids = base.index
assert ids.is_unique, "material_id is not unique in the test predictions"
y_true = base["y_true"].to_numpy()
formulas = base["formula_pretty"].to_numpy()

pred_map = {}
for fs, model in configs:
    s = get_pred(fs, model).reindex(ids)
    assert s["y_pred"].notna().all(), f"{fs}/{model} does not cover the same test rows"
    assert np.allclose(s["y_true"].to_numpy(), y_true)
    pred_map[(fs, model)] = s["y_pred"].to_numpy()

def point_metrics(y, p):
    e = p - y
    return np.abs(e).mean(), np.sqrt((e ** 2).mean()), 1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum()

for (fs, model), p in pred_map.items():
    row = results[(results.feature_set == fs) & (results.model == model)].iloc[0]
    mae, rmse, r2 = point_metrics(y_true, p)
    assert np.isclose(mae, row["Test_MAE"], atol=1e-6), (fs, model, "MAE mismatch")
    assert np.isclose(rmse, row["Test_RMSE"], atol=1e-6), (fs, model, "RMSE mismatch")
    assert np.isclose(r2, row["Test_R2"], atol=1e-6), (fs, model, "R2 mismatch")
print(f"Recomputed test metrics match Stage 6 for all {len(pred_map)} configurations "
      f"({len(y_true)} test rows, {len(set(formulas))} test formulas).")


In [ ]:
codes, uniques = pd.factorize(formulas)
G = len(uniques)

cols = {
    "n": np.bincount(codes, minlength=G).astype(float),
    "sy": np.bincount(codes, weights=y_true, minlength=G),
    "syy": np.bincount(codes, weights=y_true ** 2, minlength=G),
}
for c, p in pred_map.items():
    e = p - y_true
    cols[(c, "sabs")] = np.bincount(codes, weights=np.abs(e), minlength=G)
    cols[(c, "sse")] = np.bincount(codes, weights=e ** 2, minlength=G)

keys = list(cols.keys())
key_idx = {k: i for i, k in enumerate(keys)}
M = np.column_stack([cols[k] for k in keys])

rng = np.random.default_rng(RANDOM_SEED)
parts, done = [], 0
while done < N_BOOT:
    b = min(250, N_BOOT - done)
    W = rng.multinomial(G, np.full(G, 1.0 / G), size=b).astype(np.float64)
    parts.append(W @ M)
    done += b
T = np.vstack(parts)

def boot_metrics(c):
    n = T[:, key_idx["n"]]
    sst = T[:, key_idx["syy"]] - T[:, key_idx["sy"]] ** 2 / n
    sse = T[:, key_idx[(c, "sse")]]
    return {"MAE": T[:, key_idx[(c, "sabs")]] / n, "RMSE": np.sqrt(sse / n), "R2": 1 - sse / sst}

BM = {c: boot_metrics(c) for c in pred_map}

def ci95(a):
    lo, hi = np.percentile(a, [2.5, 97.5])
    return float(lo), float(hi)

print(f"Bootstrap done: {N_BOOT} resamples over {G} test formulas.")


In [ ]:
rows = []
for fs, model in configs:
    r = results[(results.feature_set == fs) & (results.model == model)].iloc[0]
    mae_lo, mae_hi = ci95(BM[(fs, model)]["MAE"])
    r2_lo, r2_hi = ci95(BM[(fs, model)]["R2"])
    rows.append({
        "feature_set": fs, "model": model,
        "Test_MAE": r["Test_MAE"], "MAE_CI95_low": mae_lo, "MAE_CI95_high": mae_hi,
        "Test_R2": r["Test_R2"], "R2_CI95_low": r2_lo, "R2_CI95_high": r2_hi,
    })
test_ci = pd.DataFrame(rows).sort_values(["feature_set", "Test_MAE"]).reset_index(drop=True)
print(test_ci.round(4).to_string(index=False))
test_ci.to_csv("stage7_test_metrics_with_ci.csv", index=False)


## 5. Do structural descriptors add predictive value?

This is secondary research question 1. For each model, we compare the composition+structure version with the composition-only version, two ways:

- **Cross-validation:** the change in CV MAE, alongside the fold-to-fold standard deviations for context.
- **Test set, paired bootstrap:** the change in test MAE and R2 with a 95% interval, and the share of bootstrap resamples in which adding structure gave the lower error.

Negative MAE change = structure helped. If a 95% interval includes zero, the test set alone cannot distinguish the two versions.


In [ ]:
def contrast(cA, cB, metric):
    d = BM[cA][metric] - BM[cB][metric]
    return ci95(d), float(np.mean(d < 0))

def test_point(c, metric):
    r = results[(results.feature_set == c[0]) & (results.model == c[1])].iloc[0]
    return r[f"Test_{metric}"]

rows = []
for model in ["Ridge", "HistGradientBoosting", "RandomForest", "ExtraTrees"]:
    co, cs = ("composition_only", model), ("composition_structure", model)
    r_co = results[(results.feature_set == co[0]) & (results.model == model)].iloc[0]
    r_cs = results[(results.feature_set == cs[0]) & (results.model == model)].iloc[0]
    (mae_lo, mae_hi), share_better = contrast(cs, co, "MAE")
    (r2_lo, r2_hi), _ = contrast(cs, co, "R2")
    rows.append({
        "model": model,
        "CV_MAE_comp_only": r_co["CV_MAE"], "CV_MAE_comp_struct": r_cs["CV_MAE"],
        "CV_MAE_change": r_cs["CV_MAE"] - r_co["CV_MAE"],
        "CV_std_comp_only": r_co["CV_MAE_std"], "CV_std_comp_struct": r_cs["CV_MAE_std"],
        "Test_MAE_change": test_point(cs, "MAE") - test_point(co, "MAE"),
        "Test_MAE_change_CI95_low": mae_lo, "Test_MAE_change_CI95_high": mae_hi,
        "Test_R2_change": test_point(cs, "R2") - test_point(co, "R2"),
        "Test_R2_change_CI95_low": r2_lo, "Test_R2_change_CI95_high": r2_hi,
        "share_resamples_structure_lower_MAE": share_better,
    })
structure_effect = pd.DataFrame(rows)
print(structure_effect.round(4).T.to_string(header=False))
structure_effect.to_csv("stage7_structure_effect.csv", index=False)


## 6. Selected model versus the alternatives (paired, same feature set)

How much better is the selected model than each competitor on the test set, with a 95% interval? Negative MAE difference = the selected model has the lower error.


In [ ]:
sel_fs, sel_model = SELECTED
rows = []
for model in ["Ridge", "HistGradientBoosting", "RandomForest", "ExtraTrees"]:
    if model == sel_model:
        continue
    other = (sel_fs, model)
    (lo, hi), share_better = contrast(SELECTED, other, "MAE")
    rows.append({
        "selected": f"{sel_model} ({sel_fs})", "compared_with": f"{model} ({sel_fs})",
        "MAE_difference": test_point(SELECTED, "MAE") - test_point(other, "MAE"),
        "CI95_low": lo, "CI95_high": hi,
        "share_resamples_selected_lower_MAE": share_better,
    })
vs_others = pd.DataFrame(rows)
print(vs_others.round(4).to_string(index=False))
vs_others.to_csv("stage7_selected_vs_others.csv", index=False)


## 7. Cost, robustness and interpretability

Cost and robustness columns are computed from the Stage 6 results. **The interpretability column is a qualitative judgement, not a computed quantity.** It describes what analysis tools are available for each model, and interpretability of *predictive* importance is in any case not causal (Stage 8).


In [ ]:
QUALITATIVE = {
    "Dummy (mean)": "Not applicable (reference only)",
    "Ridge": "High: one coefficient per standardised feature; assumes linear effects",
    "HistGradientBoosting": "Moderate: permutation importance / SHAP; native missing-value handling",
    "RandomForest": "Moderate: permutation importance / SHAP available",
    "ExtraTrees": "Moderate: same tools as Random Forest",
}

d = results[results.feature_set == sel_fs].copy()
d["R2_gap_train_minus_test"] = d["Train_R2"] - d["Test_R2"]
d["interpretability (qualitative)"] = d["model"].map(QUALITATIVE)
summary = d[["model", "Test_MAE", "CV_MAE_std", "R2_gap_train_minus_test", "fit_seconds",
             "interpretability (qualitative)"]].sort_values("Test_MAE")
summary.columns = ["Model", "Test MAE (eV)", "CV MAE fold std", "R2 gap (train-test)",
                   "Final fit time (s)", "Interpretability (qualitative)"]
print(f"Feature set: {sel_fs}")
print(summary.round(4).to_string(index=False))
summary.to_csv("stage7_cost_robustness_summary.csv", index=False)


## 8. Effect of the split strategy

From Stage 6: one model (HistGradientBoosting on composition+structure) was scored on a plain random split and on the formula-grouped split. A random split places some compounds' same-formula polymorphs in training, so it can flatter the model. The two test sets contain different rows, so this is an illustration of the effect and not a controlled paired comparison. Formula-grouping also does not remove chemically *similar* compounds from the training set, so even the grouped scores may overstate performance on genuinely new chemistry.


In [ ]:
print(f"Share of random-test rows whose formula also appears in random-train: "
      f"{config['random_split_formula_overlap_fraction']:.1%}\n")
print(rvg.round(4).to_string(index=False))


## 9. Figures

Two figures: (a) train, CV and test MAE for every model, which makes the overfitting pattern visible, and (b) a parity plot for the selected model (predicted vs. computed band gap on the held-out test set).


In [ ]:
os.makedirs("figures", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
width = 0.26
x = np.arange(len(MODEL_ORDER))
for ax, fs in zip(axes, FEATURE_SETS):
    d = results[results.feature_set == fs].set_index("model").loc[MODEL_ORDER]
    ax.bar(x - width, d["Train_MAE"], width, label="Train", color="#8DA0CB")
    ax.bar(x, d["CV_MAE"], width, label="CV (grouped 5-fold)", color="#66C2A5")
    ax.bar(x + width, d["Test_MAE"], width, label="Test (held-out)", color="#FC8D62")
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("HistGradientBoosting", "HistGB").replace(" (mean)", "") for m in MODEL_ORDER],
                       rotation=20)
    ax.set_title(fs.replace("_", " "))
    ax.set_xlabel("Model")
axes[0].set_ylabel("MAE (eV)")
axes[0].legend()
fig.suptitle("Train, cross-validation and test MAE by model")
plt.tight_layout()
plt.savefig("figures/fig_stage7_mae_train_cv_test.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
p_sel = pred_map[SELECTED]
r_sel = results[(results.feature_set == sel_fs) & (results.model == sel_model)].iloc[0]

fig, ax = plt.subplots(figsize=(6.2, 5.6))
hb = ax.hexbin(y_true, p_sel, gridsize=70, bins="log", cmap="viridis", mincnt=1)
lim = max(y_true.max(), p_sel.max())
ax.plot([0, lim], [0, lim], color="red", linestyle="--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Materials Project (DFT-PBE) band gap (eV)")
ax.set_ylabel("Predicted band gap (eV)")
ax.set_title(f"{sel_model} ({sel_fs.replace('_', ' ')}), held-out test set\n"
             f"MAE = {r_sel['Test_MAE']:.3f} eV, R2 = {r_sel['Test_R2']:.3f}")
ax.legend(loc="upper left")
fig.colorbar(hb, label="Count (log scale)")
plt.tight_layout()
plt.savefig("figures/fig_stage7_parity_selected_model.png", dpi=200, bbox_inches="tight")
plt.show()


## 10. Save outputs

Writes a README-ready results table (Stage 2 layout), the selected-model record, and copies everything, including the figures, to Google Drive.


In [ ]:
def df_to_markdown(df, fmt="{:.3f}"):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, r in df.iterrows():
        cells = [fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

md_parts = ["# Model comparison (errors in eV)\n"]
for fs in FEATURE_SETS:
    md_parts.append(f"\n## {fs.replace('_', ' ')}\n")
    md_parts.append(df_to_markdown(comparison_table(fs)))
with open("stage7_results_table.md", "w") as f:
    f.write("\n".join(md_parts) + "\n")

sel_ci = test_ci[(test_ci.feature_set == sel_fs) & (test_ci.model == sel_model)].iloc[0]
selected_record = {
    "selection_rule": "lowest mean grouped-CV MAE across all 10 configurations (test set not used for selection)",
    "feature_set": sel_fs, "model": sel_model,
    "CV_MAE": float(r_sel["CV_MAE"]), "CV_MAE_fold_std": float(r_sel["CV_MAE_std"]),
    "CV_R2": float(r_sel["CV_R2"]),
    "test_MAE": float(r_sel["Test_MAE"]),
    "test_MAE_CI95": [float(sel_ci["MAE_CI95_low"]), float(sel_ci["MAE_CI95_high"])],
    "test_R2": float(r_sel["Test_R2"]),
    "test_R2_CI95": [float(sel_ci["R2_CI95_low"]), float(sel_ci["R2_CI95_high"])],
    "runner_up": f"{runner['model']} ({runner['feature_set']})",
    "runner_up_CV_MAE": float(runner["CV_MAE"]),
    "bootstrap": f"{N_BOOT} resamples over test formulas; captures test-sampling variability only",
    "hyperparameters": "fixed and untuned (see Stage 6)",
}
with open("stage7_selected_model.json", "w") as f:
    json.dump(selected_record, f, indent=2)
print(json.dumps(selected_record, indent=2))

OUT_FILES = [
    "stage7_results_table.md", "stage7_selected_model.json", "stage7_test_metrics_with_ci.csv",
    "stage7_structure_effect.csv", "stage7_selected_vs_others.csv", "stage7_overfitting_table.csv",
    "stage7_cost_robustness_summary.csv",
]
FIG_FILES = ["figures/fig_stage7_mae_train_cv_test.png", "figures/fig_stage7_parity_selected_model.png"]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(os.path.join(DRIVE_DIR, "figures"), exist_ok=True)
    for fname in OUT_FILES:
        shutil.copy(fname, os.path.join(DRIVE_DIR, fname))
        print(f"Copied {fname} to Drive")
    for fname in FIG_FILES:
        shutil.copy(fname, os.path.join(DRIVE_DIR, "figures", os.path.basename(fname)))
        print(f"Copied {fname} to Drive")
except ImportError:
    print("Not running in Colab -- files remain in the local working directory.")


### Stage 7 ends here.

Models are compared and one is selected using cross-validation, with uncertainty intervals on the test results. **No feature importance, error-by-chemistry breakdown, or chemical interpretation has been done.** That is Stages 8 and 9.

The large tree models were not saved in Stage 6, so Stage 8 will refit the selected model (a short run for ExtraTrees) before computing importances.

**Do not run Stage 8 code in this notebook.** Await confirmation before proceeding.
